In [0]:
spark.sql("DROP SCHEMA IF EXISTS ecommerce.gold CASCADE")
spark.sql("CREATE SCHEMA ecommerce.gold")

In [0]:
from pyspark.sql.functions import *

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS ecommerce.gold")

In [0]:
orders = spark.read.table("ecommerce.silver.orders")
customers = spark.read.table("ecommerce.silver.customers")
items = spark.read.table("ecommerce.silver.order_items")
products = spark.read.table("ecommerce.silver.products")
payments = spark.read.table("ecommerce.silver.payments")
reviews = spark.read.table("ecommerce.silver.reviews")
sellers = spark.read.table("ecommerce.silver.sellers")
geolocation = spark.read.table("ecommerce.silver.geolocation")

In [0]:
dim_customer = customers.select(
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state"
).dropDuplicates()

dim_customer.write.mode("overwrite").saveAsTable("ecommerce.gold.dim_customer")
display(dim_customer)

In [0]:
dim_product = products.select(
    "product_id",
    "product_category_name",
    "missing_category_flag"
).dropDuplicates()

dim_product.write.mode("overwrite").saveAsTable("ecommerce.gold.dim_product")

In [0]:
dim_seller = sellers.select(
    "seller_id",
    "seller_city",
    "seller_state"
).dropDuplicates()

dim_seller.write.mode("overwrite").saveAsTable("ecommerce.gold.dim_seller")

In [0]:
dim_geo = geolocation.select(
    col("geolocation_city").alias("city"),
    col("geolocation_state").alias("state")
).dropDuplicates()

dim_geo.write.mode("overwrite").saveAsTable("ecommerce.gold.dim_geography")

In [0]:
dim_date = orders.select("order_purchase_timestamp") \
    .withColumn("date", to_date("order_purchase_timestamp")) \
    .withColumn("year", year("order_purchase_timestamp")) \
    .withColumn("month", month("order_purchase_timestamp")) \
    .dropDuplicates()

dim_date.write.mode("overwrite").saveAsTable("ecommerce.gold.dim_date")

In [0]:
fact_orders = orders.select(
    "order_id", "customer_id", "order_purchase_timestamp",
    "order_status", "is_late_delivery"
)

fact_orders.write.mode("overwrite").saveAsTable("ecommerce.gold.fact_orders")


In [0]:
display(fact_orders)

In [0]:
fact_items = items.select(
    "order_id",
    "product_id",
    "seller_id",
    "price",
    "freight_value"
)

fact_items.write.mode("overwrite").saveAsTable("ecommerce.gold.fact_order_items")

In [0]:
fact_payments = payments.select(
    "order_id",
    "total_payment"
)

fact_payments.write.mode("overwrite").saveAsTable("ecommerce.gold.fact_payments")

In [0]:
fact_reviews = reviews.select(
    "order_id",
    "review_score"
)

fact_reviews.write.mode("overwrite").saveAsTable("ecommerce.gold.fact_reviews")

In [0]:
fact_sales_summary = fact_orders \
    .join(fact_items, "order_id") \
    .join(fact_payments, "order_id") \
    .join(dim_product, "product_id") \
    .join(dim_seller, "seller_id") \
    .join(dim_customer, "customer_id") \
    .join(fact_reviews, "order_id", "left")

In [0]:
fact_sales_summary = fact_sales_summary.select(
    "order_id",
    "customer_id",
    "product_id",
    "seller_id",
    "customer_city",
    "customer_state",
    "product_category_name",
    "price",
    "freight_value",
    "total_payment",
    "review_score",
    "order_purchase_timestamp",
    "order_status",
    "is_late_delivery"
)

In [0]:

fact_sales_summary = fact_sales_summary \
    .withColumn("year", year("order_purchase_timestamp")) \
    .withColumn("month", month("order_purchase_timestamp"))

fact_sales_summary.write.mode("overwrite") \
    .saveAsTable("ecommerce.gold.fact_sales_summary")


In [0]:
fs = spark.read.table("ecommerce.gold.fact_sales_summary")
display(fs)

In [0]:
order_level = fs.select(
    "order_id", "customer_id", "total_payment",
    "order_status", "is_late_delivery",
    "customer_city", "customer_state",
    "year", "month"
).dropDuplicates(["order_id"])

display(fs)

In [0]:
fs.groupBy("year", "month") \
    .agg(sum("total_payment").alias("revenue")) \
    .write.mode("overwrite").saveAsTable("ecommerce.gold.kpi_revenue_month")

In [0]:
display(kpi_revenue_month)

In [0]:
order_level.agg(avg("total_payment").alias("avg_order_value")) \
    .write.mode("overwrite").saveAsTable("ecommerce.gold.kpi_aov")

In [0]:
display(kpi_aov)

In [0]:
kpi_top_products = fs.groupBy("product_id") \
    .agg(sum("price").alias("total_revenue")) \
    .orderBy(desc("total_revenue")).limit(10)

kpi_top_products.write.mode("overwrite").saveAsTable("ecommerce.gold.kpi_top_products")

In [0]:
display(kpi_top_products)

In [0]:
fs.groupBy("seller_id") \
    .agg(sum("price").alias("revenue")) \
    .orderBy(desc("revenue")).limit(10) \
    .write.mode("overwrite").saveAsTable("ecommerce.gold.kpi_top_sellers")

In [0]:
display(kpi_top_sellers)

In [0]:
order_level.groupBy("customer_city", "customer_state") \
    .count().write.mode("overwrite").saveAsTable("ecommerce.gold.kpi_orders_geo")

In [0]:
display(kpi_orders_geo)

In [0]:
order_level.groupBy("customer_id", "year", "month").count() \
    .withColumn("customer_type", when(col("count")==1,"new").otherwise("returning")) \
    .write.mode("overwrite").saveAsTable("ecommerce.gold.kpi_customer_type")

In [0]:
display(kpi_customer_type)

In [0]:
order_level.groupBy("customer_id") \
    .agg(sum("total_payment").alias("clv")) \
    .write.mode("overwrite").saveAsTable("ecommerce.gold.kpi_clv")

In [0]:
display(kpi_clv)

In [0]:
fulfilled = order_level.filter(col("order_status")=="delivered").count()
total_orders = order_level.count()

spark.createDataFrame([(fulfilled/total_orders,)], ["fulfillment_rate"]) \
    .write.mode("overwrite").saveAsTable("ecommerce.gold.kpi_fulfillment")


In [0]:
display(kpi_fulfillment)

In [0]:
late = order_level.filter(col("is_late_delivery")==True).count()

kpi_late_delivery = spark.createDataFrame([(late/total_orders,)], ["late_delivery_percentage"])
kpi_late_delivery.write.mode("overwrite").saveAsTable("ecommerce.gold.kpi_late_delivery")

In [0]:
display(kpi_late_delivery)

In [0]:
kpi_delivery_time = orders.withColumn("delivery_days",
    datediff("order_delivered_customer_date","order_purchase_timestamp")) \
    .agg(avg("delivery_days").alias("avg_delivery_days"))

kpi_delivery_time.write.mode("overwrite").saveAsTable("ecommerce.gold.kpi_delivery_time")

In [0]:
display(kpi_delivery_time)

In [0]:
kpi_review_product = fs.groupBy("product_id") \
    .agg(avg("review_score").alias("avg_review"))

kpi_review_product.write.mode("overwrite").saveAsTable("ecommerce.gold.kpi_review_product")

In [0]:
display(kpi_review_product)

In [0]:
kpi_review_seller = fs.groupBy("seller_id") \
    .agg(avg("review_score").alias("avg_review"))

kpi_review_seller.write.mode("overwrite").saveAsTable("ecommerce.gold.kpi_review_seller")

In [0]:
display(kpi_review_seller)

In [0]:
low = fs.filter(col("review_score") < 3).count()

kpi_low_rating = spark.createDataFrame([(low/total_orders,)], ["low_rating_percentage"])
kpi_low_rating.write.mode("overwrite").saveAsTable("ecommerce.gold.kpi_low_rating")

In [0]:
display(kpi_low_rating)